- v69 `LB=74.32`
    - Model
        - Include almost all features
        - paramter change :  `cat_l2=0.1`, `num_leaves=256`, `stopping_rounds=50`, `reg_sqrt=rue`

- v65 `LB=74.94`
    - Model
        - Data preprocessing pipeline: Add routine to fill missing time.
        - Added new features `lag_2_day_consumption_counterpart`, `lag_7_day_mean_consumption_counterpart`, `euros_per_mwh`,
        - paramter change :  `stopping_rounds=100`

- v60 `LB=74.40`
    - Model
        - Changed base_lag to 48 hours


- v59 `LB=75.59`
    - Model
        - Removed forecast filter for `hour_ahead`
        - paramter change :  `cat_l2=0.1`


- v56 `LB=75.73`
    - Model
        - seperate models & features for is_consumption
        - cv_agg's l1: 29.0241 + 12.5268, cv_agg's l1: 47.618 + 4.03412
        - 



- v51 `LB=76.53`
    - Model
        - cv_agg's l1: 39.5448 + 7.62686
        - Added feature 'daytype`
        - parameter chagne: `cat_l2=1`, `reg_sqrt=True`
    - Analysis
        - Inspect errors by feature values


- v35 `LB=78.1`
    - Tweedie loss
    - Removed features: `sin_month`, `cos_month`
    - Added features: `lag_1_day_1h`, `lag_1_day_1h`, `lag_1_day_2h`, `lag_1_day_3h`


- v34 `LB 81.8`
    - Fix lag features for test time inference


In [1]:
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor
import catboost as cb
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error
import datetime

In [2]:
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt

In [3]:
import json
with open("/kaggle/input/predict-energy-behavior-of-prosumers/county_id_to_name_map.json") as file:
    county_codes = json.load(file)

In [4]:
# county_coordinates from Google results
county_coordinates = {
    0: [59.3334, 25.2467],
    1: [58.9240, 22.5919],
    2: [59.2593, 27.4137],
    3: [58.8867, 25.5001],
    4: [58.7970, 26.5977],
    5: [59.3019, 26.3280],
    6: [58.9723, 23.8741],
    7: [58.5262, 24.4020],
    8: [58.0670, 27.0739],
    9: [59.0001, 24.8049],
    10: [58.4850, 22.6136],
    11: [58.4057, 26.8016],
    12: [0.0, 0.0],
    13: [57.9103, 26.1602],
    14: [58.2822, 25.5752],
    15: [57.7377, 27.1399],
}

In [5]:
import numpy as np
import pandas as pd

train = pd.read_csv("/kaggle/input/predict-energy-behavior-of-prosumers/train.csv")
elec_prices = pd.read_csv("/kaggle/input/predict-energy-behavior-of-prosumers/electricity_prices.csv")
gas_prices_train = pd.read_csv("/kaggle/input/predict-energy-behavior-of-prosumers/gas_prices.csv")
client_train = pd.read_csv("/kaggle/input/predict-energy-behavior-of-prosumers/client.csv")

train.datetime = pd.to_datetime(train.datetime)#.dt.date
train['date'] = pd.to_datetime(train.datetime).dt.date

client_train['date'] = pd.to_datetime(client_train.date).dt.date

# elec_prices.forecast_date = pd.to_datetime(elec_prices.forecast_date).dt.date
# gas_prices.forecast_date = pd.to_datetime(gas_prices.forecast_date).dt.date
train.head(2)

   county  is_business  product_type  target  is_consumption   datetime  \
0       0            0             1   0.713               0 2021-09-01   
1       0            0             1  96.590               1 2021-09-01   

   data_block_id  row_id  prediction_unit_id        date  
0              0       0                   0  2021-09-01  
1              0       1                   0  2021-09-01  

In [6]:
client_features = ['eic_count', 'installed_capacity']

In [7]:
# historical_weather_train = pd.read_csv("/kaggle/input/predict-energy-behavior-of-prosumers/historical_weather.csv")
# historical_weather_train

In [8]:
forecast_weather_train = pd.read_csv("/kaggle/input/predict-energy-behavior-of-prosumers/forecast_weather.csv")

In [9]:
weather_features = ["dewpoint", 
#                     "cloudcover_high", 
                    "cloudcover_low", 
                    "cloudcover_mid", 
                    "cloudcover_total", 
                    "10_metre_u_wind_component",
                    "10_metre_v_wind_component",
                    "direct_solar_radiation", 
                    "surface_solar_radiation_downwards", 
#                     "snowfall",
                   ]

In [10]:
def process_forecast(forecast_weather_df):
    
    available_coordinates = forecast_weather_df[['latitude', 'longitude']].drop_duplicates().values
    
    
#     forecast_weather_df = forecast_weather_df[forecast_weather_df['hours_ahead'] > 24]
    
    
    forecast_weather_df['datetime'] = pd.to_datetime(forecast_weather_df.forecast_datetime).dt.tz_convert(None)
    forecast_weather_df = forecast_weather_df.set_index(['latitude', 'longitude'])

    forecast_weather_df['county'] = 999

    for key, coordinates in county_coordinates.items():
        nearest_point = ((coordinates - available_coordinates) ** 2).sum(axis=1).argmin()
#         print(coordinates, available_coordinates[nearest_point], nearest_point)

        forecast_weather_df.loc[tuple(available_coordinates[nearest_point]), 'county'] = key
    
    
    forecast_weather_df = forecast_weather_df[forecast_weather_df['county'] != 999]
    forecast_weather_df = forecast_weather_df.groupby(['county', 'datetime'])[weather_features].mean()
    forecast_weather_df = forecast_weather_df.reset_index()
    return forecast_weather_df

In [11]:
elec_prices

             forecast_date  euros_per_mwh          origin_date  data_block_id
0      2021-09-01 00:00:00          92.51  2021-08-31 00:00:00              1
1      2021-09-01 01:00:00          88.90  2021-08-31 01:00:00              1
2      2021-09-01 02:00:00          87.35  2021-08-31 02:00:00              1
3      2021-09-01 03:00:00          86.88  2021-08-31 03:00:00              1
4      2021-09-01 04:00:00          88.43  2021-08-31 04:00:00              1
...                    ...            ...                  ...            ...
15281  2023-05-30 19:00:00          82.10  2023-05-29 19:00:00            637
15282  2023-05-30 20:00:00         150.85  2023-05-29 20:00:00            637
15283  2023-05-30 21:00:00          82.10  2023-05-29 21:00:00            637
15284  2023-05-30 22:00:00          82.09  2023-05-29 22:00:00            637
15285  2023-05-30 23:00:00          -1.29  2023-05-29 23:00:00            637

[15286 rows x 4 columns]

In [12]:
elec_prices.forecast_date = pd.to_datetime(elec_prices.forecast_date)
elec_prices.origin_date = pd.to_datetime(elec_prices.origin_date)
elec_prices['data_available_datetime'] = elec_prices.forecast_date + datetime.timedelta(days=1)

In [13]:
elec_prices['data_available_date'] = elec_prices['data_available_datetime'].dt.date
daily_elec_prices = elec_prices.groupby(['data_available_date'])[['euros_per_mwh']].mean().reset_index()
daily_elec_prices = daily_elec_prices.rename(columns={"euros_per_mwh": "daily_euros_per_mwh"})
daily_elec_prices['daily_euros_per_mwh_lag_1_day'] = daily_elec_prices['daily_euros_per_mwh'].shift(1)
# daily_elec_prices['daily_euros_per_mwh_lag_2_day'] = daily_elec_prices['daily_euros_per_mwh'].shift(2)
# daily_elec_prices['daily_euros_per_mwh_lag_3_day'] = daily_elec_prices['daily_euros_per_mwh'].shift(3)
daily_elec_prices['daily_euros_per_mwh_lag_7_day'] = daily_elec_prices['daily_euros_per_mwh'].shift(7)
daily_elec_prices['daily_euros_per_mwh_rolling_7_day_mean'] =  daily_elec_prices['daily_euros_per_mwh'].rolling(7).mean()

elec_prices = elec_prices.merge(
                    daily_elec_prices,
                    on=['data_available_date'],
                    how='left'
                )

In [14]:
gas_prices_train.plot(x='forecast_date', y='lowest_price_per_mwh')

<Axes: xlabel='forecast_date'>

<Figure size 640x480 with 1 Axes>

In [15]:
def propcess_gas_prices(gas_prices_df):

    new_df = gas_prices_df.copy()
    new_df['forecast_date'] = pd.to_datetime(new_df['forecast_date']).dt.date
    new_df['data_available_date'] = new_df.forecast_date + datetime.timedelta(days=1)
    new_df['gas_mid_price'] = (new_df['lowest_price_per_mwh'] + new_df['highest_price_per_mwh']) /2
    
    new_df['gas_mid_price_lag_1_day'] = new_df['gas_mid_price'].shift(1)

    new_df['gas_mid_price_lag_7_day'] = new_df['gas_mid_price'].shift(7)
    new_df['gas_mid_price_rolling_7_day_mean'] =  new_df['gas_mid_price'].rolling(7).mean()
    
    return new_df
    

In [16]:
train = train.merge(
    elec_prices[[
        "euros_per_mwh", 
         "daily_euros_per_mwh",
         'daily_euros_per_mwh_lag_1_day',
         'daily_euros_per_mwh_lag_7_day',
         'daily_euros_per_mwh_rolling_7_day_mean',
         "data_available_datetime",
        ]],
    how="left",
    right_on="data_available_datetime",
    left_on="datetime",
   )

train = train.merge(
    propcess_gas_prices(gas_prices_train)[[
        "data_available_date",
        "gas_mid_price",
        'gas_mid_price_lag_1_day',
        'gas_mid_price_lag_7_day',
        'gas_mid_price_rolling_7_day_mean',
        ]],
    how="left",
    right_on="data_available_date",
    left_on="date",
   )



train = train.merge(
    client_train.drop("date", axis=1), 
    on=["county", "is_business", "product_type", "data_block_id"], 
    how="left")


train = train.merge(
    process_forecast(forecast_weather_train),
    how="left",
    on=["county", "datetime",],
    )


/tmp/ipykernel_20/1931801343.py:18: PerformanceWarning: indexing past lexsort depth may impact performance.
  forecast_weather_df.loc[tuple(available_coordinates[nearest_point]), 'county'] = key


In [17]:
def fill_missing_time(df, start_time=None, end_time=None, inference=False):
    
    if start_time is None:
        start_time = df['datetime'].min()
        
    if end_time is None:
        end_time = df['datetime'].max()
        
    df_array = np.empty((16, 2, 4, 2), dtype=object)
    idx = pd.date_range(start_time, end_time, freq='H')
    
    for county in range(16):
        for is_business in [0, 1]:
            for product_type in [0, 1, 2, 3]:
                for is_consumption in [0, 1]:

#                     print(county, is_business, product_type, is_consumption, end="")
                    try:
                        temp_df = df[
                                    (df['county']==county) *
                                    (df['is_business']==is_business) *
                                    (df['product_type']==product_type) *
                                    (df['is_consumption']==is_consumption)
                                    ].set_index('datetime')
                        length = len(temp_df)
                
                        if inference and length==0:
                            continue

                        temp_df = temp_df.reindex(
                            idx, 
#                             method='nearest',
                        )
                        temp_df[["county", "is_business", "product_type", "is_consumption"]] = [county, is_business, product_type, is_consumption]
                        df_array[county, is_business, product_type, is_consumption] = temp_df

                            
#                         print(len(df_array[county, is_business, product_type, is_consumption]))
                    except:
#                         print(0)
                        pass

                for is_consumption in [0, 1]:
                    if df_array[county, is_business, product_type, 1-is_consumption] is None:
                        continue
                    df_array[county, is_business, product_type, is_consumption]['consumption_counterpart'] = df_array[county, is_business, product_type, 1-is_consumption]['target']
            
                
    new_df = pd.concat([e for e in df_array.flatten() if e is not None])
    new_df.index.names = ['datetime']
    new_df = new_df.reset_index()
#     new_df = new_df.set_index(['county', 'is_business', 'product_type', 'is_consumption', 'datetime']).sort_index(ascending=True)
    return new_df, df_array

In [18]:
%%time
train, df_arr = fill_missing_time(train)

CPU times: user 16.3 s, sys: 2.6 s, total: 18.8 s
Wall time: 18.9 s


In [19]:
train = train.set_index(['county', 'is_business', 'product_type', 'is_consumption', 'datetime']).sort_index(ascending=True)

base_lag = 48

train['lag_2_day'] = train.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].shift(24*2)
train['lag_3_day'] = train.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].shift(24*3)
train['lag_4_day'] = train.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].shift(24*4)
train['lag_5_day'] = train.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].shift(24*5)
train['lag_6_day'] = train.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].shift(24*6)
train['lag_7_day'] = train.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].shift(24*7)
train['lag_8_day'] = train.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].shift(24*8)

train['lag_7_day_mean'] = np.mean([train[f'lag_{i}_day'] for i in range(2,9)], axis=0)

train['lag_2_day_1h'] = train.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].shift(48+1)
train['lag_3_day_1h'] = train.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].shift(72+1)
train['lag_4_day_1h'] = train.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].shift(96+1)


train['rolling_mean_24'] = train.reset_index().groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].rolling(24).mean().shift(base_lag).values

# train['rolling_mean_24x7'] = train.reset_index().groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].rolling(24*7).mean().shift(base_lag).values
train['rolling_std_24'] = train.reset_index().groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].rolling(24).std().shift(base_lag).values


train['lag_2_day_consumption_counterpart'] = train.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['consumption_counterpart'].shift(24*2)
train['lag_3_day_consumption_counterpart'] = train.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['consumption_counterpart'].shift(24*3)
train['lag_4_day_consumption_counterpart'] = train.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['consumption_counterpart'].shift(24*4)
train['lag_5_day_consumption_counterpart'] = train.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['consumption_counterpart'].shift(24*5)
train['lag_6_day_consumption_counterpart'] = train.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['consumption_counterpart'].shift(24*6)
train['lag_7_day_consumption_counterpart'] = train.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['consumption_counterpart'].shift(24*7)
train['lag_8_day_consumption_counterpart'] = train.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['consumption_counterpart'].shift(24*8)

train['lag_7_day_mean_consumption_counterpart'] = np.mean([train[f'lag_{i}_day_consumption_counterpart'] for i in range(2,9)], axis=0)

train = train.reset_index()

In [20]:
train = train[train['datetime'] < pd.to_datetime("2023-05-28 00:00:00")]


train.dropna(axis=0, how='any', inplace=True) # filter out missing values
print(int(100*len(train)/2018352),"% of dataset")
train.head()

96 % of dataset


       county  is_business  product_type  is_consumption            datetime  \
30816       0            0             1               0 2021-09-09 00:00:00   
30817       0            0             1               0 2021-09-09 01:00:00   
30818       0            0             1               0 2021-09-09 02:00:00   
30819       0            0             1               0 2021-09-09 03:00:00   
30820       0            0             1               0 2021-09-09 04:00:00   

       target  data_block_id   row_id  prediction_unit_id        date  ...  \
30816   0.058            8.0  23424.0                 0.0  2021-09-09  ...   
30817   0.086            8.0  23546.0                 0.0  2021-09-09  ...   
30818   0.016            8.0  23668.0                 0.0  2021-09-09  ...   
30819   0.000            8.0  23790.0                 0.0  2021-09-09  ...   
30820   0.000            8.0  23912.0                 0.0  2021-09-09  ...   

       rolling_mean_24  rolling_std_24  lag_2_day_

In [21]:
national_holidays =[
                    [2021, 1, 1],
                    [2021, 2, 24],
                    [2021, 4, 2],
                    [2021, 4, 4],
                    [2021, 5, 1],
                    [2021, 5, 23],
                    [2021, 6, 23],
                    [2021, 6, 24],
                    [2021, 8, 20],
                    [2021, 12, 24],
                    [2021, 12, 25],
                    [2021, 12, 26],


                    [2022, 1, 1],
                    [2022, 2, 24],
                    [2022, 4, 15],
                    [2022, 4, 17],
                    [2022, 5, 1],
                    [2022, 6, 5],
                    [2022, 6, 23],
                    [2022, 6, 24],
                    [2022, 8, 20],
                    [2022, 12, 24],
                    [2022, 12, 25],
                    [2022, 12, 26],

                    [2023, 1, 1],
                    [2023, 2, 24],
                    [2023, 4, 7],
                    [2023, 4, 9],
                    [2023, 5, 1],
                    [2023, 5, 28],
                    [2023, 6, 23],
                    [2023, 6, 24],
                    [2023, 8, 20],
                    [2023, 12, 24],
                    [2023, 12, 25],
                    [2023, 12, 26],

                    [2024, 1, 1],
                    [2024, 2, 24],
                    [2024, 3, 29],
                    [2024, 3, 31],
                    [2024, 5, 1],
                    [2024, 5, 19],
                    [2024, 6, 23],
                    [2024, 6, 24],
                    [2024, 8, 20],
                    [2024, 12, 24],
                    [2024, 12, 25],
                    [2024, 12, 26],
]

In [22]:
import datetime
national_holidays_dt = [pd.to_datetime('-'.join(str(n) for n in holiday)).date() for holiday in national_holidays]

def append_day_type(train_df):
    train_df['days_in_month'] = train_df['datetime'].dt.days_in_month

    train_df['inner_weekday'] = (train_df['datetime'].dt.dayofweek >= 1) * (train_df['datetime'].dt.dayofweek <= 3) # Tuesday to Thursday
    train_df['holiday'] = (train_df['datetime'].dt.dayofweek >= 5) * (train_df['datetime'].dt.dayofweek <= 6) # Saturday and Sunday
    train_df['pre_holiday'] = train_df['datetime'].dt.dayofweek == 4
    train_df['post_holiday'] = train_df['datetime'].dt.dayofweek == 0
    
    train_df['national_holiday'] = train_df['datetime'].dt.date.isin(national_holidays_dt)
    train_df['pre_national_holiday'] = train_df['datetime'].dt.date.isin(d-datetime.timedelta(days=1) for d in national_holidays_dt)
    train_df['post_national_holiday'] = train_df['datetime'].dt.date.isin(d+datetime.timedelta(days=1) for d in national_holidays_dt)
    
    train_df['daytype'] = None
    train_df.loc[train_df['inner_weekday'], 'daytype'] = 'inner_weekday'
    train_df.loc[train_df['pre_holiday'] + train_df['pre_national_holiday'], 'daytype'] = 'pre_holiday'
    train_df.loc[train_df['post_holiday'] + train_df['post_national_holiday'], 'daytype'] = 'post_holiday'
    train_df.loc[train_df['holiday'] + train_df['national_holiday'], 'daytype'] = 'holiday'
    

    
    train_df['daytype'] = train_df['daytype'].astype(dtype)
    
    fts_to_remove = ['inner_weekday', 'holiday', 'pre_holiday', 'post_holiday', 'national_holiday', 'pre_national_holiday', 'post_national_holiday']
    for ft in fts_to_remove:
        del train_df[ft]

    
    return train_df

In [23]:
# dtype = str
dtype = "category"

train['hour'] = train['datetime'].dt.hour
train['dayofweek'] = train['datetime'].dt.dayofweek.astype(dtype)
train["month"]  = train["datetime"].dt.month

train = append_day_type(train)

hours_in_day = 24
train['sin_hour'] = np.sin(2*np.pi*train.hour/hours_in_day)
train['cos_hour'] = np.cos(2*np.pi*train.hour/hours_in_day)

months_in_year = 12
train['sin_month'] = np.sin(2*np.pi*train.month/months_in_year)
train['cos_month'] = np.cos(2*np.pi*train.month/months_in_year)


train['county'] = train['county'].astype(dtype)
train['product_type'] = train['product_type'].astype(dtype)

In [24]:
raw_features = ["county",
                "is_business",
                "product_type",
                "is_consumption",
                "dayofweek",
                "daytype",
#                 "sin_month",
#                 "cos_month",
                "sin_hour",
                "cos_hour",
                "lag_2_day",
                "lag_3_day",
                "lag_4_day",
                "lag_7_day_mean",
                "lag_5_day",
                "lag_6_day",
                "lag_7_day",
                "lag_2_day_1h",
#                 "lag_3_day_1h",
#                 "lag_4_day_1h",
                "rolling_mean_24",
#                 "rolling_mean_24x7",
                "rolling_std_24",
                "lag_2_day_consumption_counterpart",
#                 "lag_3_day_consumption_counterpart",
#                 "lag_4_day_consumption_counterpart",
#                 "lag_5_day_consumption_counterpart",
#                 "lag_6_day_consumption_counterpart",
                "lag_7_day_consumption_counterpart",
                "lag_7_day_mean_consumption_counterpart",
                "euros_per_mwh",
#                 "daily_euros_per_mwh", 
#                 'daily_euros_per_mwh_lag_1_day',
#                 'daily_euros_per_mwh_lag_7_day',
#                 'daily_euros_per_mwh_rolling_7_day_mean',
                # Gas price
#                 "gas_mid_price",
#                 'gas_mid_price_lag_1_day',
#                 'gas_mid_price_lag_7_day',
#                 'gas_mid_price_rolling_7_day_mean',
               ]


In [25]:
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error as MAE
from sklearn.model_selection import GroupKFold

In [26]:
train.reset_index(drop=True, inplace=True)
# X = train.loc[train[train['hour'] == 0].index][features]
# Y = np.array([train['target'][train[train['hour'] == 0].index + i].values for i in range(24)]).T
# X.shape, Y.shape

In [27]:
def select_features(raw_features, weather_features, client_features, is_consumption, verbose=0):
    
    features = raw_features + weather_features + client_features
    features.remove('is_consumption')
#     features.remove('is_business')
    
    fts_to_remove = []
    fts_to_add = []
#     if is_consumption==0:
#         fts_to_remove = [
#             "rolling_mean_24", 
#             "rolling_std_24", 
# #             "cloudcover_high", 
# #            "cloudcover_low", 
# #            "cloudcover_mid", 
#            "cloudcover_total", 
# #            "snowfall",
#             "lag_2_day_consumption_counterpart",
#             "lag_7_day_mean_consumption_counterpart",
#         ]
        
#     else:
#         fts_to_remove = [
#             "cloudcover_high", 
#             "cloudcover_low", 
#             "cloudcover_mid", 
#             "cloudcover_total", 
#             "10_metre_u_wind_component",
#             "10_metre_v_wind_component",
#             "snowfall",
# #             "lag_5_day",
#         ]

#     if is_consumption==1 and is_business==1:
#         fts_to_add += [
#                 'daily_euros_per_mwh_lag_1_day',
#                 'daily_euros_per_mwh_lag_7_day', 
#                 'daily_euros_per_mwh_rolling_7_day_mean',
                
                # Gas price
#                 'gas_mid_price_lag_1_day',
#                 'gas_mid_price_lag_7_day',
#                 'gas_mid_price_rolling_7_day_mean',
#         ]
    
    if verbose==1:
        print(f'removing features: {fts_to_remove}')
        print(f'Adding features: {fts_to_add}')
        
    for ft in fts_to_remove:
        features.remove(ft)
        

    features += fts_to_add
        
    return features

In [28]:
def get_monotone_constraints_map(features, is_consumption, is_business=0):
    
    monotone_constraints_map = {ft: 0 for ft in features}
    
    if is_consumption ==1:
        monotone_constraints_map['euros_per_mwh'] = -1
        monotone_constraints_map['eic_count'] = 1

    else:
        monotone_constraints_map['euros_per_mwh'] = 1
        monotone_constraints_map['installed_capacity'] = 1
        monotone_constraints_map["direct_solar_radiation"] = 1
        monotone_constraints_map["cloudcover_total"] = -1
        
    
    return monotone_constraints_map

In [29]:
results = {}
for is_consumption in [0, 1]:
#     for is_business in [0, 1]:
        
#         key = f"is_consumption={is_consumption}, is_business={is_business}"
        key = f"is_consumption={is_consumption}"
        print(f"key: {key}")
        
        train_subset = train[
            (train['is_consumption'] == is_consumption)
#             * (train['is_business'] == is_business)
        ]
        
        train_subset.reset_index(drop=True, inplace=True)

        features = select_features(raw_features, weather_features, client_features, is_consumption=is_consumption, verbose=1)

        X = train_subset[features]
        Y = train_subset['target']

        G = train_subset['month']
        gkf = GroupKFold(n_splits=5, )
        folds = gkf.split(X, Y, G)
        
#         tscv = TimeSeriesSplit(n_splits=5, )        
#         folds = tscv.split(X, Y, G)
        
        lgb_train = lgb.Dataset(X, Y)


        # categories = ['county','product_type', 'dayofweek']
        # categories = ['county','product_type', 'daytype']
        categories = ['county','product_type', 'daytype', 'dayofweek']

        params = {
            'boosting_type': 'gbdt',
            'objective': 'tweedie',
            'metric': 'mae', 
            'learning_rate': 0.1,
            'feature_fraction': 0.9, 
            'cat_l2': 0.1,
    #         'reg_sqrt': True,
            'max_depth': 7,
            'num_leaves': 127,
            'monotone_constraints': [value for key, value in get_monotone_constraints_map(features, is_consumption).items()],
            'monotone_constraints_method': 'advanced',
    #         'monotone_penalty': 0.01,
        #     'subsample ': 0.9,
        #     'reg_alpha':0.2, 
        #     'reg_lambda':0.2, 
        #     'extra_trees': True,
            'verbose': -1,
        }

        results[key] = lgb.cv(
            params,
            lgb_train,
            categorical_feature=categories, 
            folds=folds,
            num_boost_round=3000,
            callbacks=[
                lgb.early_stopping(stopping_rounds=50, verbose=True),
                lgb.log_evaluation(10)
            ],
            return_cvbooster=True
        )

key: is_consumption=0
removing features: []
Adding features: []


/opt/conda/lib/python3.10/site-packages/lightgbm/basic.py:2068: UserWarning: categorical_feature in Dataset is overridden.
New categorical_feature is ['county', 'dayofweek', 'daytype', 'product_type']
  _log_warning('categorical_feature in Dataset is overridden.\n'


Training until validation scores don't improve for 50 rounds
[10]	cv_agg's l1: 67.1709 + 30.8587
[20]	cv_agg's l1: 43.7037 + 23.0518
[30]	cv_agg's l1: 33.4282 + 17.0379
[40]	cv_agg's l1: 29.7571 + 14.7979
[50]	cv_agg's l1: 28.7786 + 14.5261
[60]	cv_agg's l1: 28.333 + 14.4209
[70]	cv_agg's l1: 28.1191 + 14.3152
[80]	cv_agg's l1: 28.0096 + 14.296
[90]	cv_agg's l1: 27.8602 + 14.2282
[100]	cv_agg's l1: 27.7731 + 14.1512
[110]	cv_agg's l1: 27.7121 + 14.0889
[120]	cv_agg's l1: 27.6795 + 14.047
[130]	cv_agg's l1: 27.6637 + 14.0518
[140]	cv_agg's l1: 27.6392 + 14.017
[150]	cv_agg's l1: 27.6183 + 13.9866
[160]	cv_agg's l1: 27.607 + 14.0156
[170]	cv_agg's l1: 27.5782 + 13.9867
[180]	cv_agg's l1: 27.6027 + 14.0134
[190]	cv_agg's l1: 27.6022 + 13.99
[200]	cv_agg's l1: 27.5978 + 13.9842
[210]	cv_agg's l1: 27.5957 + 13.9732
[220]	cv_agg's l1: 27.5897 + 13.9298
Early stopping, best iteration is:
[171]	cv_agg's l1: 27.5778 + 13.9869
key: is_consumption=1
removing features: []
Adding features: []


/opt/conda/lib/python3.10/site-packages/lightgbm/basic.py:2068: UserWarning: categorical_feature in Dataset is overridden.
New categorical_feature is ['county', 'dayofweek', 'daytype', 'product_type']
  _log_warning('categorical_feature in Dataset is overridden.\n'


Training until validation scores don't improve for 50 rounds
[10]	cv_agg's l1: 249.161 + 25.2699
[20]	cv_agg's l1: 129.317 + 14.2472
[30]	cv_agg's l1: 77.5615 + 6.49996
[40]	cv_agg's l1: 61.1313 + 3.96001
[50]	cv_agg's l1: 56.994 + 3.60467
[60]	cv_agg's l1: 55.5522 + 3.67047
[70]	cv_agg's l1: 55.0808 + 3.81832
[80]	cv_agg's l1: 54.9674 + 3.94771
[90]	cv_agg's l1: 54.7482 + 3.96133
[100]	cv_agg's l1: 54.583 + 3.94935
[110]	cv_agg's l1: 54.4735 + 4.03342
[120]	cv_agg's l1: 54.3679 + 4.10692
[130]	cv_agg's l1: 54.269 + 4.14464
[140]	cv_agg's l1: 54.2222 + 4.20358
[150]	cv_agg's l1: 54.166 + 4.1951
[160]	cv_agg's l1: 54.1193 + 4.2168
[170]	cv_agg's l1: 54.0524 + 4.19421
[180]	cv_agg's l1: 53.987 + 4.21696
[190]	cv_agg's l1: 53.9143 + 4.2311
[200]	cv_agg's l1: 53.8926 + 4.28327
[210]	cv_agg's l1: 53.8322 + 4.23695
[220]	cv_agg's l1: 53.7564 + 4.23448
[230]	cv_agg's l1: 53.7092 + 4.19106
[240]	cv_agg's l1: 53.6585 + 4.22162
[250]	cv_agg's l1: 53.593 + 4.22977
[260]	cv_agg's l1: 53.559 + 4.24

- cv_agg's l1: 28.3732 + 13.9627
- cv_agg's l1: 27.0412 + 14.2798
- cv_agg's l1: 14.922 + 3.59908
- cv_agg's l1: 83.8011 + 2.87022


- cv_agg's l1: 28.9376 + 14.3825
- cv_agg's l1: 27.3628 + 14.4734
- cv_agg's l1: 15.6974 + 3.92428
- cv_agg's l1: 83.7568 + 2.81861



- cv_agg's l1: 27.6637 + 14.0748
- cv_agg's l1: 53.0345 + 3.94255


- cv_agg's l1: 28.8601 + 16.6712
- cv_agg's l1: 53.7149 + 3.60032


In [30]:
import lightgbm

for key in results.keys():
    print(key)
    importance = np.mean([booster.feature_importance(importance_type='gain') / booster.feature_importance(importance_type='gain').sum()
                     for booster in results[key]['cvbooster'].boosters], axis=0) * 100
    feature_name = results[key]['cvbooster'].boosters[0].feature_name()
    tuples = sorted(zip(feature_name, importance), key=lambda x: x[1], reverse=True)
    
    plt.figure(figsize=(10,8))
    ax = sns.barplot(pd.DataFrame(tuples), x=1, y=0, orient='h')
    for i in ax.containers:
        ax.bar_label(i,)
    plt.show()

is_consumption=0


<Figure size 1000x800 with 1 Axes>

is_consumption=1


<Figure size 1000x800 with 1 Axes>

In [31]:
error_dfs = []

for i, (train_index, test_index) in enumerate(gkf.split(X, Y, G)):
    print(f"Fold {i}: Validation Month: {G[test_index].unique()}")
    
    
    model = results[key]["cvbooster"].boosters[i]
    
    preds = model.predict(train_subset.iloc[test_index][features])
    targets = train_subset.iloc[test_index]['target']
#     pred = preds.mean(axis=0)
    mae = abs(targets - preds)
    print(f"MAE_SCORE = {mae.mean():.3f}")
    
    error_df = train_subset.iloc[test_index][features].copy()
    error_df['MAE'] = mae
    error_df['ME'] = preds - targets
    
    error_dfs.append(error_df)
    
agg_error_df = pd.concat(error_dfs)
    
for ft in ['county', 'product_type', 'daytype', 'dayofweek',]:
# for ft in ['county', 'is_consumption', 'is_business', 'product_type',]:
    plt.figure(figsize=(9,4))
    sns.boxenplot(
        agg_error_df[[ft, 'ME']], x=ft, y='ME', outlier_prop=0.03,showfliers=False, k_depth='proportion',
    )
    plt.show()

    print(f"feature: {ft}")
    for feature_value in agg_error_df[ft].unique():
        me = agg_error_df[agg_error_df[ft]==feature_value]['ME'].mean()
        mae = abs(agg_error_df[agg_error_df[ft]==feature_value]['ME']).mean()
        print(f"\tfeature_value: {feature_value}, ME {me:.2f}, MAE {mae:.2f}")
            


Fold 0: Validation Month: [1 7 8]
MAE_SCORE = 52.425
Fold 1: Validation Month: [ 9 10  6]
MAE_SCORE = 46.736
Fold 2: Validation Month: [12  2]
MAE_SCORE = 56.993
Fold 3: Validation Month: [11  4]
MAE_SCORE = 51.048
Fold 4: Validation Month: [3 5]
MAE_SCORE = 57.282


<Figure size 900x400 with 1 Axes>

feature: county
	feature_value: 0, ME -11.07, MAE 132.20
	feature_value: 1, ME 0.23, MAE 6.55
	feature_value: 2, ME -9.36, MAE 70.61
	feature_value: 3, ME -0.03, MAE 27.04
	feature_value: 4, ME -0.96, MAE 40.80
	feature_value: 5, ME -2.11, MAE 54.74
	feature_value: 6, ME -0.04, MAE 25.96
	feature_value: 7, ME -2.24, MAE 52.21
	feature_value: 8, ME 0.49, MAE 14.10
	feature_value: 9, ME 0.15, MAE 16.29
	feature_value: 10, ME -1.67, MAE 30.36
	feature_value: 11, ME -0.27, MAE 94.96
	feature_value: 12, ME 0.80, MAE 53.74
	feature_value: 13, ME -2.70, MAE 21.11
	feature_value: 14, ME -1.84, MAE 58.94
	feature_value: 15, ME -0.27, MAE 28.18


<Figure size 900x400 with 1 Axes>

feature: product_type
	feature_value: 1, ME -0.08, MAE 18.31
	feature_value: 2, ME -1.71, MAE 9.11
	feature_value: 3, ME -3.75, MAE 84.32
	feature_value: 0, ME -7.63, MAE 75.24


<Figure size 900x400 with 1 Axes>

feature: daytype
	feature_value: holiday, ME -1.99, MAE 47.26
	feature_value: post_holiday, ME -1.94, MAE 56.56
	feature_value: inner_weekday, ME -2.86, MAE 53.79
	feature_value: pre_holiday, ME -3.17, MAE 56.78


<Figure size 900x400 with 1 Axes>

feature: dayofweek
	feature_value: 5, ME -3.04, MAE 48.36
	feature_value: 6, ME -2.68, MAE 43.50
	feature_value: 0, ME -2.72, MAE 55.68
	feature_value: 1, ME -2.94, MAE 57.62
	feature_value: 2, ME -2.06, MAE 54.21
	feature_value: 3, ME -2.89, MAE 53.90
	feature_value: 4, ME -1.25, MAE 55.47


In [32]:
# import shap
# shap.initjs()

# shap_values = shap.TreeExplainer(results[key]["cvbooster"].boosters[0]).shap_values(train_subset.iloc[test_index][features])
# shap.summary_plot(shap_values, train_subset.iloc[test_index][features])

In [33]:
for is_consumption in [0, 1]:
#     for is_business in [0, 1]:
#         key = f"is_consumption={is_consumption}, is_business={is_business}"
        key = f"is_consumption={is_consumption}"
        print("CV_MAE_SCORE =", results[key]["l1-mean"][-1])

# Predition of whole training data
# preds = np.array([model.predict(train[features]) for model in results["cvbooster"].boosters])
# pred = preds.mean(axis=0)
# print("MAE_SCORE =", MAE(train['target'], pred))

CV_MAE_SCORE = 27.577789258766796
CV_MAE_SCORE = 52.86681443006614


# Make pred & Submission

In [34]:
def append_lag(test, placeholder_previous_day, historic_train, predicting_start_time, truncate=True,):
    
    new_df = pd.concat([test, placeholder_previous_day, historic_train,])
    new_df, _ = fill_missing_time(new_df, start_time=placeholder_previous_day['datetime'].min()-datetime.timedelta(days=30), inference=True)
    
    new_df = new_df.set_index(['county', 'is_business', 'product_type', 'is_consumption', 'datetime']).sort_index(ascending=True)

    new_df['lag_2_day'] = new_df.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].shift(24*2)
    new_df['lag_3_day'] = new_df.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].shift(24*3)
    new_df['lag_4_day'] = new_df.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].shift(24*4)
    new_df['lag_5_day'] = new_df.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].shift(24*5)
    new_df['lag_6_day'] = new_df.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].shift(24*6)
    new_df['lag_7_day'] = new_df.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].shift(24*7)
    new_df['lag_8_day'] = new_df.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].shift(24*8)
    
    new_df['lag_7_day_mean'] = np.mean([new_df[f'lag_{i}_day'] for i in range(2,9)], axis=0) 

    new_df['lag_2_day_1h'] = new_df.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].shift(48+1)
    new_df['lag_3_day_1h'] = new_df.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].shift(72+1)
    new_df['lag_4_day_1h'] = new_df.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].shift(96+1)
    
    new_df['rolling_mean_24'] = new_df.reset_index().groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].rolling(24).mean().shift(base_lag).values
#     new_df['rolling_mean_24x7'] = new_df.reset_index().groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].rolling(24*7).mean().shift(24).values
    new_df['rolling_std_24'] = new_df.reset_index().groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].rolling(24).std().shift(base_lag).values

    new_df['lag_2_day_consumption_counterpart'] = new_df.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['consumption_counterpart'].shift(24*2)
    new_df['lag_3_day_consumption_counterpart'] = new_df.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['consumption_counterpart'].shift(24*3)
    new_df['lag_4_day_consumption_counterpart'] = new_df.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['consumption_counterpart'].shift(24*4)
    new_df['lag_5_day_consumption_counterpart'] = new_df.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['consumption_counterpart'].shift(24*5)
    new_df['lag_6_day_consumption_counterpart'] = new_df.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['consumption_counterpart'].shift(24*6)
    new_df['lag_7_day_consumption_counterpart'] = new_df.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['consumption_counterpart'].shift(24*7)
    new_df['lag_8_day_consumption_counterpart'] = new_df.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['consumption_counterpart'].shift(24*8)

    new_df['lag_7_day_mean_consumption_counterpart'] = np.mean([new_df[f'lag_{i}_day_consumption_counterpart'] for i in range(2,9)], axis=0)
    
    new_df = new_df.reset_index()
    if truncate:
        new_df = new_df[new_df['datetime'] >= predicting_start_time]
    
    return new_df[new_df['row_id'].notna()]

In [35]:
historic_train = train[
    (train['datetime'] >= pd.to_datetime("2023-04-28 00:00:00")) * (train['datetime'] < pd.to_datetime("2023-05-26 00:00:00"))
]


import enefit
env = enefit.make_env()

iter_test = env.iter_test()

sample_prediction_last = None
sample_prediction_last_last = None

for (test, revealed_targets, client, historical_weather,
        forecast_weather, electricity_prices, gas_prices, sample_prediction) in iter_test:
    
    if sample_prediction_last_last is not None:
        print("MAE: ", abs(revealed_targets['target'] - sample_prediction_last_last['target']).mean())
    
    
#     test = test_ori.copy()
    
    test=test.rename(columns={'prediction_datetime':'datetime'})
    
    predicting_start_time = test['datetime'].min()
    print(f"Predicting start time: {predicting_start_time}")
    
    test['is_business'] = test['is_business'].astype(int)
    test['is_consumption'] = test['is_consumption'].astype(int)
    test['county'] = test['county'].astype(int)
    test['product_type'] = test['product_type'].astype(int)


    # Electricity Prices Data
    electricity_prices.forecast_date = pd.to_datetime(electricity_prices.forecast_date)
    electricity_prices['data_available_datetime'] = electricity_prices.forecast_date + datetime.timedelta(days=1)
    
    electricity_prices['data_available_date'] = electricity_prices['data_available_datetime'].dt.date
    daily_electricity_prices = electricity_prices.groupby(['data_available_date'])[['euros_per_mwh']].mean().reset_index()
    daily_electricity_prices = daily_electricity_prices.rename(columns={"euros_per_mwh": "daily_euros_per_mwh"})

    electricity_prices = electricity_prices.merge(
                        daily_electricity_prices,
                        on=['data_available_date'],
                        how='left'
                    )

    test = test.merge(electricity_prices[["euros_per_mwh", "daily_euros_per_mwh", "data_available_datetime",]],
                how="left",
                right_on="data_available_datetime",
                left_on="datetime",
               )
    
    # Client Data
    client['date'] = pd.to_datetime(client.date).dt.date
    client['is_business'] = client['is_business'].astype(int)
    test = test.merge(
        client, 
        on=["county", "is_business", "product_type"], how="left")
    
    
    # Weahter Forecast
    test = test.merge(
        process_forecast(forecast_weather),
        how="left",
        on=["county", "datetime",],
    )

    
    test['hour'] = test['datetime'].dt.hour
    test["month"]  = test["datetime"].dt.month

    test['sin_hour'] = np.sin(2*np.pi*test.hour/hours_in_day)
    test['cos_hour'] = np.cos(2*np.pi*test.hour/hours_in_day)
    
    test['sin_month'] = np.sin(2*np.pi*test.month/months_in_year)
    test['cos_month'] = np.cos(2*np.pi*test.month/months_in_year)


    historic_train = pd.concat([historic_train, revealed_targets])
    
    placeholder_previous_day = revealed_targets.copy()
    placeholder_previous_day['datetime'] += datetime.timedelta(days=1)
    placeholder_previous_day['target'] = np.nan
    
    test['target'] = np.nan
    test = append_lag(test, placeholder_previous_day, historic_train, predicting_start_time=predicting_start_time)

    test['county'] = test['county'].astype(dtype)
    test['product_type'] = test['product_type'].astype(dtype)
    test['dayofweek'] = test['datetime'].dt.dayofweek.astype(dtype)
    test = append_day_type(test)

    test = test.sort_values('row_id', ascending=True)
    
    
    test['pred'] = None
    for is_consumption in [0, 1]:
#         for is_business in [0, 1]:
            features = select_features(raw_features, weather_features, client_features, is_consumption=is_consumption)

            test_subset = test[
                (test['is_consumption'] == is_consumption)
#                 * (test['is_business'] == is_business)
            ]
            
#             key = f"is_consumption={is_consumption}, is_business={is_business}"
            key = f"is_consumption={is_consumption}"
            pred = np.array([model.predict(test_subset[features]) for model in results[key]["cvbooster"].boosters]).mean(axis=0).clip(min=0)

            test.loc[
                (test['is_consumption'] == is_consumption),
#                 * (test['is_business'] == is_business), 
                'pred'] = pred
    

    sample_prediction['target'] = test['pred'].reset_index(drop=True).values.astype(float)
    
    sample_prediction_last_last = sample_prediction_last
    sample_prediction_last = sample_prediction
    
    env.predict(sample_prediction)



This version of the API is not optimized and should not be used to estimate the runtime of your code on the hidden test set.
Predicting start time: 2023-05-28 00:00:00


/tmp/ipykernel_20/1931801343.py:18: PerformanceWarning: indexing past lexsort depth may impact performance.
  forecast_weather_df.loc[tuple(available_coordinates[nearest_point]), 'county'] = key


Predicting start time: 2023-05-29 00:00:00


/tmp/ipykernel_20/1931801343.py:18: PerformanceWarning: indexing past lexsort depth may impact performance.
  forecast_weather_df.loc[tuple(available_coordinates[nearest_point]), 'county'] = key


MAE:  65.09375052887201
Predicting start time: 2023-05-30 00:00:00


/tmp/ipykernel_20/1931801343.py:18: PerformanceWarning: indexing past lexsort depth may impact performance.
  forecast_weather_df.loc[tuple(available_coordinates[nearest_point]), 'county'] = key


MAE:  53.85990631191236
Predicting start time: 2023-05-31 00:00:00


/tmp/ipykernel_20/1931801343.py:18: PerformanceWarning: indexing past lexsort depth may impact performance.
  forecast_weather_df.loc[tuple(available_coordinates[nearest_point]), 'county'] = key


- MAE:  68.354636037987668.3546360379876
- MAE:  58.0373678676262258.03736786762622

In [36]:
test.isna().sum(axis=0)

county                                    0
is_business                               0
product_type                              0
is_consumption                            0
datetime                                  0
                                         ..
lag_7_day_mean_consumption_counterpart    0
dayofweek                                 0
days_in_month                             0
daytype                                   0
pred                                      0
Length: 63, dtype: int64